# SSTW validation_scale 正式门禁单 Notebook 入口

该 Notebook 用于在 Colab 冷启动环境中一次性执行 `validation_scale` 正式门禁测试。它会挂载 Google Drive、拉取仓库代码、安装依赖、执行真实 Wan2.1 runtime、runtime attack / detection、现代 external baseline adapter、内部消融、replay/sketch 或 Claim-3 downgrade、统计 CI、artifact rebuild 和 `validation_scale_gate`。

重要边界:

1. Notebook 只是远程执行入口, 不手写正式 records、tables、figures、reports 或 manifests。
2. 样本数量、run_root、package 目录和阶段开关全部来自 `configs/paper_workflow/generative_video_notebook_workflows.json`。
3. 该 Notebook 只允许 `validation_scale`, 不用于 `pilot_paper` 或未来 `full_paper`。
4. 若 Google Drive 中没有已通过的 motion threshold artifact, 默认会提前失败。只有显式设置 `SSTW_RUN_MOTION_CALIBRATION_IF_MISSING=true` 时才会先运行 motion calibration。
5. 现代视频水印 baseline 必须配置真实官方命令或真实 SSTW 外层命令, 不允许使用伪分数闭合门禁。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import subprocess
import sys
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 1.1 可编辑 workflow profile 切换
# 该单 Notebook 只用于 validation_scale 正式门禁测试。
# 不要把这里改成 pilot_paper 或 full_paper; pilot_paper 应使用同构 paper 协议但独立 profile。
import os

SSTW_WORKFLOW_PROFILE_VALUE = 'validation_scale'

if SSTW_WORKFLOW_PROFILE_VALUE.strip():
    os.environ['SSTW_WORKFLOW_PROFILE'] = SSTW_WORKFLOW_PROFILE_VALUE.strip()
    print('SSTW_WORKFLOW_PROFILE =', os.environ['SSTW_WORKFLOW_PROFILE'])
else:
    raise RuntimeError('validation_scale formal gate Notebook 必须显式使用 validation_scale。')


In [ ]:
# 2. 冷启动获取仓库代码
# Colab 环境不会保留代码, 每次运行都需要 clone 或 pull。
REPO_URL = os.environ.get('SSTW_REPO_URL', 'https://github.com/RICHAAARC/SSTW.git')
REPO_DIR = os.environ.get('SSTW_REPO_DIR', '/content/SSTW')
REPO_REF = os.environ.get('SSTW_REPO_REF', '').strip()

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 SSTW_REPO_URL, 或将仓库上传到 /content/SSTW。')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"
if REPO_REF:
    print('切换到 SSTW_REPO_REF =', REPO_REF)
    !git fetch --all --tags
    !git checkout "$REPO_REF"
!git rev-parse --short HEAD


In [ ]:
# 3. 安装 Colab GPU 运行依赖
# 第三方 external baseline 的额外依赖由各 baseline 官方命令自行安装或在命令中处理。
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf huggingface_hub


In [ ]:
# 4. 可选 Hugging Face 认证
# token 只写入当前 Colab 环境变量, 不写入 records、tables、reports 或 package manifest。
try:
    from huggingface_hub import login
except Exception as exc:
    raise RuntimeError('huggingface_hub 未安装或无法导入, 请重新执行依赖安装单元。') from exc

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
# 5. 读取统一 workflow profile 配置并初始化 Drive 布局
from paper_workflow.notebook_utils import generative_video_model_probe_workflow as probe_workflow

NOTEBOOK_ROLE = 'validation_scale_formal_gate'
WORKFLOW_PROFILE = os.environ.get('SSTW_WORKFLOW_PROFILE') or probe_workflow.default_workflow_profile_for_notebook_role(NOTEBOOK_ROLE)
workflow_profile = probe_workflow.resolve_notebook_workflow_profile(WORKFLOW_PROFILE, NOTEBOOK_ROLE)
PROFILE = workflow_profile['runtime_profile']
layout = probe_workflow.ensure_drive_layout(
    DRIVE_PROJECT_ROOT,
    workflow_profile=WORKFLOW_PROFILE,
    notebook_role=NOTEBOOK_ROLE,
)

print(json.dumps({
    'workflow_profile': workflow_profile,
    'layout': layout,
    'enabled_stage_plan': probe_workflow.build_workflow_stage_plan(WORKFLOW_PROFILE, NOTEBOOK_ROLE),
}, ensure_ascii=False, indent=2))


In [ ]:
# 6. 通用执行 helper
# Notebook 只调用仓库模块, 不在 cell 中手写正式 records、tables、figures 或 reports。
def stage_enabled(stage_name: str) -> bool:
    return probe_workflow.workflow_stage_enabled(WORKFLOW_PROFILE, NOTEBOOK_ROLE, stage_name)


def run_or_raise(stage_name: str, command: list[str]) -> None:
    print('\n===== stage:', stage_name, '=====')
    print(' '.join(command))
    result = probe_workflow.run_command(command)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


def read_json_artifact(path: Path) -> dict:
    if not path.exists():
        return {'artifact_status': 'missing', 'path': str(path)}
    text = path.read_text(encoding='utf-8-sig')
    return json.loads(text)


In [ ]:
# 7. 可编辑运行参数
# 样本数量、阶段计划和输出目录不要在这里硬编码, 它们由 workflow profile 配置控制。
MODEL_ID = os.environ.get('SSTW_MODEL_ID', 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers')
CROSS_MODEL_ID = os.environ.get('SSTW_CROSS_MODEL_ID', '')
SEMANTIC_MODEL_ID = os.environ.get('SSTW_SEMANTIC_MODEL_ID', 'openai/clip-vit-base-patch32')
SEMANTIC_FRAME_LIMIT = int(os.environ.get('SSTW_SEMANTIC_FRAME_LIMIT', '8'))
DISABLE_SEMANTIC_METRIC = os.environ.get('SSTW_DISABLE_SEMANTIC_METRIC', 'false').lower() == 'true'
INCLUDE_VIDEOS_IN_PACKAGE = os.environ.get('SSTW_INCLUDE_VIDEOS_IN_PACKAGE', 'true').lower() == 'true'
RUN_MOTION_CALIBRATION_IF_MISSING = os.environ.get('SSTW_RUN_MOTION_CALIBRATION_IF_MISSING', 'false').lower() == 'true'

# run-through test 默认开启, 目的是让 Notebook 可以直接跑完整工程链路并落盘阻断原因。
# 该模式不会伪造正式 external baseline 分数, validation_scale_gate 仍会如实 FAIL。
# 若要执行严格正式门禁, 请配置 6 个真实官方 baseline command 后设置为 false。
VALIDATION_SCALE_RUN_THROUGH_TEST = os.environ.get('SSTW_VALIDATION_SCALE_RUN_THROUGH_TEST', 'true').lower() == 'true'

# external baseline 配置。
# 推荐路径 A: 保持 bridge 开启, 配置 SSTW_<BASELINE>_OFFICIAL_EVAL_COMMAND。
# 可选路径 B: 直接配置 SSTW_<BASELINE>_EVAL_COMMAND, 该外层命令必须写出 {output_json_path}。
BASELINE_COMMAND_TEMPLATES = {}
RUN_EXTERNAL_BASELINE_SOURCE_CLONE = os.environ.get('SSTW_RUN_EXTERNAL_BASELINE_SOURCE_CLONE', 'true').lower() == 'true'
REQUIRE_MODERN_BASELINE_COMMANDS_FOR_PAPER_GATE = os.environ.get('SSTW_REQUIRE_MODERN_BASELINE_COMMANDS_FOR_PAPER_GATE', 'true').lower() == 'true'
USE_MODERN_BASELINE_BRIDGE_COMMANDS = os.environ.get('SSTW_USE_MODERN_BASELINE_BRIDGE_COMMANDS', 'true').lower() == 'true'
REQUIRE_MODERN_BASELINE_BRIDGE_OFFICIAL_COMMANDS = os.environ.get('SSTW_REQUIRE_MODERN_BASELINE_BRIDGE_OFFICIAL_COMMANDS', 'true').lower() == 'true'
EXTERNAL_BASELINE_EVIDENCE_PATHS = [
    item for item in os.environ.get('SSTW_EXTERNAL_BASELINE_EVIDENCE_PATHS', '').split(os.pathsep)
    if item
]

# 示例: 使用 repository bridge 时, 配置某个官方 baseline 内部命令。
# os.environ['SSTW_SPDMARK_OFFICIAL_EVAL_COMMAND'] = (
#     'python /content/SSTW/external_baseline/primary/spdmark/source/<official_eval_script>.py '
#     '--source-video {source_video_path} '
#     '--attacked-video {attacked_video_path} '
#     '--attack-name {attack_name} '
#     '--output-json {official_output_json_path}'
# )

print(json.dumps({
    'model_id': MODEL_ID,
    'cross_model_id': CROSS_MODEL_ID,
    'semantic_model_id': SEMANTIC_MODEL_ID,
    'semantic_frame_limit': SEMANTIC_FRAME_LIMIT,
    'disable_semantic_metric': DISABLE_SEMANTIC_METRIC,
    'include_videos_in_package': INCLUDE_VIDEOS_IN_PACKAGE,
    'run_motion_calibration_if_missing': RUN_MOTION_CALIBRATION_IF_MISSING,
    'validation_scale_run_through_test': VALIDATION_SCALE_RUN_THROUGH_TEST,
    'run_external_baseline_source_clone': RUN_EXTERNAL_BASELINE_SOURCE_CLONE,
    'use_modern_baseline_bridge_commands': USE_MODERN_BASELINE_BRIDGE_COMMANDS,
}, ensure_ascii=False, indent=2))


In [ ]:
# 8. 现代视频水印 baseline command 预检
# validation_scale 是 paper gate 前最后门禁, 必须在真实 GPU 生成前检查 baseline command 配置。
# run-through test 模式只跳过 Notebook 前置中断, 不改变 preflight FAIL artifact, 也不支持正式论文 claim。
modern_command_env = {}
if stage_enabled('external_baseline_colab_preflight'):
    modern_command_config_summary = probe_workflow.write_modern_baseline_colab_command_config_summary(
        layout,
        profile=WORKFLOW_PROFILE,
    )
    print(json.dumps(modern_command_config_summary, ensure_ascii=False, indent=2))
    command_template_source = {}
    if USE_MODERN_BASELINE_BRIDGE_COMMANDS:
        command_template_source.update(probe_workflow.build_modern_baseline_official_bridge_command_templates(PROFILE))
    command_template_source.update(dict(os.environ))
    command_template_source.update(BASELINE_COMMAND_TEMPLATES)
    bridge_preflight_decision = probe_workflow.write_modern_baseline_official_bridge_preflight_decision(
        layout,
        profile=WORKFLOW_PROFILE,
        command_env=command_template_source,
        use_bridge_commands=USE_MODERN_BASELINE_BRIDGE_COMMANDS,
        require_bridge_official_commands=REQUIRE_MODERN_BASELINE_BRIDGE_OFFICIAL_COMMANDS,
    )
    print(json.dumps(bridge_preflight_decision, ensure_ascii=False, indent=2))
    if bridge_preflight_decision.get('external_baseline_official_bridge_preflight_decision') == 'FAIL' and VALIDATION_SCALE_RUN_THROUGH_TEST:
        print('validation-scale run-through test: bridge 官方命令缺失, 将继续运行并在后续 artifacts 中保留 FAIL 证据。')
    probe_workflow.validate_modern_baseline_official_bridge_for_profile(
        bridge_preflight_decision,
        allow_run_through_test=VALIDATION_SCALE_RUN_THROUGH_TEST,
    )
    modern_command_env = probe_workflow.build_modern_baseline_command_env(PROFILE, command_template_source)
    external_baseline_preflight_decision = probe_workflow.write_external_baseline_colab_preflight_decision(
        layout,
        profile=WORKFLOW_PROFILE,
        command_env=modern_command_env,
        require_modern_baseline_commands_for_paper_gate=REQUIRE_MODERN_BASELINE_COMMANDS_FOR_PAPER_GATE,
        run_external_baseline_source_clone=RUN_EXTERNAL_BASELINE_SOURCE_CLONE,
        evidence_paths=EXTERNAL_BASELINE_EVIDENCE_PATHS,
    )
    print(json.dumps(external_baseline_preflight_decision, ensure_ascii=False, indent=2))
    if external_baseline_preflight_decision.get('external_baseline_colab_preflight_decision') == 'FAIL' and VALIDATION_SCALE_RUN_THROUGH_TEST:
        print('validation-scale run-through test: 外层 baseline command 缺失, 将继续运行并在后续 artifacts 中保留 FAIL 证据。')
    probe_workflow.validate_modern_baseline_commands_for_profile(
        external_baseline_preflight_decision,
        allow_run_through_test=VALIDATION_SCALE_RUN_THROUGH_TEST,
    )
    for env_name, command_template in modern_command_env.items():
        if command_template:
            os.environ[env_name] = command_template
    os.environ['SSTW_EXTERNAL_BASELINE_EVIDENCE_PATHS'] = os.pathsep.join(EXTERNAL_BASELINE_EVIDENCE_PATHS)


In [ ]:
# 9. 检查或可选生成 motion threshold calibration artifact
# validation_scale 不能用自己的 evaluation 样本重新估计阈值, 只能复用 motion_calibration run_root 中冻结的 artifact。
calibration_decision_path = Path(layout['motion_threshold_artifact_run_root']) / 'artifacts' / 'motion_threshold_calibration_decision.json'
if not calibration_decision_path.exists():
    if not RUN_MOTION_CALIBRATION_IF_MISSING:
        raise RuntimeError(
            '缺少 motion threshold calibration artifact: '
            f'{calibration_decision_path}. 请先运行 motion_threshold_calibration_colab.ipynb, '
            '或显式设置 SSTW_RUN_MOTION_CALIBRATION_IF_MISSING=true 后重新运行本 Notebook。'
        )
    print('未发现 motion threshold artifact, 开始执行 motion_calibration 前置流程。')
    calibration_profile = 'motion_calibration'
    calibration_role = 'motion_threshold_calibration'
    calibration_layout = probe_workflow.ensure_drive_layout(
        DRIVE_PROJECT_ROOT,
        workflow_profile=calibration_profile,
        notebook_role=calibration_role,
    )
    if probe_workflow.workflow_stage_enabled(calibration_profile, calibration_role, 'prepare_prompt_suite'):
        run_or_raise('motion_calibration.prepare_prompt_suite', probe_workflow.build_prompt_suite_command(calibration_layout))
    if probe_workflow.workflow_stage_enabled(calibration_profile, calibration_role, 'wan21_runtime_generation'):
        run_or_raise(
            'motion_calibration.wan21_runtime_generation',
            probe_workflow.build_colab_runtime_command(calibration_layout, calibration_profile, MODEL_ID, CROSS_MODEL_ID),
        )
    if probe_workflow.workflow_stage_enabled(calibration_profile, calibration_role, 'formal_metric_scoring'):
        run_or_raise(
            'motion_calibration.formal_metric_scoring',
            probe_workflow.build_formal_metric_command(
                calibration_layout,
                semantic_model_id=SEMANTIC_MODEL_ID,
                semantic_frame_limit=SEMANTIC_FRAME_LIMIT,
                disable_semantic_metric=DISABLE_SEMANTIC_METRIC,
            ),
        )
    if probe_workflow.workflow_stage_enabled(calibration_profile, calibration_role, 'motion_threshold_calibration'):
        run_or_raise('motion_calibration.motion_threshold_calibration', probe_workflow.build_motion_threshold_calibration_command(calibration_layout))

if not calibration_decision_path.exists():
    raise RuntimeError(f'motion threshold calibration 运行结束后仍未发现 artifact: {calibration_decision_path}')

print(json.dumps(read_json_artifact(calibration_decision_path), ensure_ascii=False, indent=2))


In [ ]:
# 10. 生成 prompt suite、运行 Wan2.1、计算 formal metrics
if stage_enabled('prepare_prompt_suite'):
    run_or_raise('prepare_prompt_suite', probe_workflow.build_prompt_suite_command(layout))

if stage_enabled('wan21_runtime_generation'):
    run_or_raise('wan21_runtime_generation', probe_workflow.build_colab_runtime_command(layout, PROFILE, MODEL_ID, CROSS_MODEL_ID))

if stage_enabled('formal_metric_scoring'):
    run_or_raise(
        'formal_metric_scoring',
        probe_workflow.build_formal_metric_command(
            layout,
            semantic_model_id=SEMANTIC_MODEL_ID,
            semantic_frame_limit=SEMANTIC_FRAME_LIMIT,
            disable_semantic_metric=DISABLE_SEMANTIC_METRIC,
        ),
    )


In [ ]:
# 11. 复用 motion threshold artifact
if stage_enabled('motion_threshold_reuse_check'):
    motion_threshold_reuse = probe_workflow.write_motion_threshold_reuse_artifact_for_profile(layout, WORKFLOW_PROFILE)
    print(json.dumps(motion_threshold_reuse, ensure_ascii=False, indent=2))


In [ ]:
# 12. 生成主方法后处理、runtime attack、detection 和 small-scale gate 记录
runtime_stage_commands = {
    'mechanism_postprocess': probe_workflow.build_mechanism_postprocess_command,
    'pilot_matrix_postprocess': probe_workflow.build_pilot_matrix_postprocess_command,
    'runtime_attack': probe_workflow.build_runtime_attack_command,
    'runtime_detection': probe_workflow.build_runtime_detection_command,
    'small_scale_claim_pilot_gate': probe_workflow.build_small_scale_claim_pilot_gate_command,
}
for stage_name, command_builder in runtime_stage_commands.items():
    if stage_enabled(stage_name):
        run_or_raise(stage_name, command_builder(layout))


In [ ]:
# 13. 执行 external baseline source intake 与 formal comparison
# 该阶段读取同一 run_root 中已经生成的 runtime detection records, 不重新生成 Wan2.1 视频。
if stage_enabled('external_baseline_source_intake'):
    run_or_raise(
        'external_baseline_source_intake',
        probe_workflow.build_external_baseline_source_intake_command(
            layout,
            execute_clone=RUN_EXTERNAL_BASELINE_SOURCE_CLONE,
        ),
    )

if stage_enabled('external_baseline_comparison'):
    run_or_raise('external_baseline_comparison', probe_workflow.build_external_baseline_comparison_command(layout))


In [ ]:
# 14. 执行内部消融、adaptive attack、replay/sketch、CI、artifact rebuild 和 validation_scale gate
gate_stage_commands = {
    'validation_internal_ablation': probe_workflow.build_validation_internal_ablation_command,
    'adaptive_attack_proxy': probe_workflow.build_adaptive_attack_command,
    'replay_and_sketch_gate': probe_workflow.build_replay_and_sketch_gate_command,
    'claim3_downgrade_gate': probe_workflow.build_claim3_downgrade_command,
    'statistical_confidence_interval': probe_workflow.build_statistical_confidence_interval_command,
    'validation_artifact_rebuild_dry_run': probe_workflow.build_validation_artifact_rebuild_dry_run_command,
    'validation_scale_gate': probe_workflow.build_validation_scale_gate_command,
}
for stage_name, command_builder in gate_stage_commands.items():
    if stage_enabled(stage_name):
        run_or_raise(stage_name, command_builder(layout))


In [ ]:
# 15. 末尾治理审计与 Google Drive 打包
if stage_enabled('quick_tests_and_harness'):
    !pytest -q
    !python tools/harness/run_all_audits.py

if stage_enabled('drive_packaging'):
    run_or_raise('drive_packaging', probe_workflow.build_drive_packaging_command(layout, include_videos=INCLUDE_VIDEOS_IN_PACKAGE))


In [ ]:
# 16. 汇总关键落盘结果
# 这里只读取并打印 artifact 摘要, 不生成或修改正式结果。
artifact_relpaths = [
    'artifacts/external_baseline_colab_preflight_decision.json',
    'artifacts/external_baseline_official_bridge_preflight_decision.json',
    'artifacts/motion_threshold_reuse_decision.json',
    'artifacts/runtime_attack_decision.json',
    'artifacts/runtime_detection_decision.json',
    'artifacts/external_baseline_comparison_decision.json',
    'artifacts/validation_internal_ablation_decision.json',
    'artifacts/replay_and_sketch_gate_decision.json',
    'artifacts/claim3_downgrade_decision.json',
    'artifacts/statistical_confidence_interval_decision.json',
    'artifacts/validation_artifact_rebuild_dry_run_decision.json',
    'artifacts/validation_scale_gate_decision.json',
]
for relpath in artifact_relpaths:
    path = Path(layout['drive_run_root']) / relpath
    print('\n===== artifact:', relpath, '=====')
    payload = read_json_artifact(path)
    print(json.dumps(payload, ensure_ascii=False, indent=2))

package_dir = Path(layout['drive_package_dir'])
print('\npackage_dir =', package_dir)
if package_dir.exists():
    for path in sorted(package_dir.glob('*')):
        print(path)
